# 🧭 DSA Field Guide — Python Edition
### For Quant Dev / SWE interviews (intern → junior)

This notebook is meant to be **re-read**, not read once. Every section has the same shape:

| Block | What it's for |
|---|---|
| 🧠 **Mental model** | The one-paragraph picture you draw on the whiteboard first |
| ✏️ **Whiteboard sketch** | ASCII diagram you can reproduce on paper in 20 seconds |
| ⏱️ **Costs** | Big-O you should say out loud without thinking |
| 🎯 **Pattern triggers** | Phrases in a problem statement that should make you reach for this tool |
| 💻 **Code** | Clean, interview-length implementations with `assert` tests — run them, break them, rewrite them |
| ⚠️ **Gotchas** | The bugs interviewers watch for |

**How to review:** Run all cells (they should all pass). Then pick a section, *close the code*, re-implement it on paper, and compare. The last section has a random self-quiz cell.

**Contents**
0. Python interview toolkit & complexity cheat sheet
1. Arrays & strings — two pointers, sliding window, prefix sums
2. Hash maps & sets
3. Stacks, queues, monotonic stack/deque
4. Linked lists
5. Trees & BSTs
6. Heaps / priority queues
7. Binary search (including "binary search on the answer")
8. Graphs — BFS, DFS, topo sort, Dijkstra, union-find
9. Tries
10. Dynamic programming
11. Backtracking
12. Sorting, selection & intervals
13. Bit manipulation
14. Design problems — LRU cache, min stack, rate limiter
15. 📈 Quant corner — order book, VWAP, rolling median, streaming top-K
16. Whiteboard playbook, pattern router, Python pitfalls, self-quiz

---
# 0. Python Interview Toolkit

🧠 **Mental model:** In an interview, Python's standard library *is* your data-structure library. Know these imports cold and what each one costs.

| Need | Use | Notes |
|---|---|---|
| Dynamic array / stack | `list` | `append`/`pop()` O(1); `pop(0)`/`insert(0,x)` are **O(n)** |
| Queue / deque | `collections.deque` | O(1) at both ends; indexing middle is O(n) |
| Hash map | `dict`, `defaultdict`, `Counter` | O(1) average |
| Hash set | `set` | O(1) average membership |
| Min-heap | `heapq` on a list | push/pop O(log n), peek `h[0]` O(1); max-heap → negate |
| Sorted-array search | `bisect` | `bisect_left` = lower bound, O(log n); `insort` is O(n) (shifting) |
| Memoization | `functools.cache` / `lru_cache` | top-down DP in one line |
| Ordered dict w/ move-to-end | `collections.OrderedDict` | LRU cache shortcut |
| Infinity | `math.inf` / `float('inf')` | |

### Complexity of common Python operations

| Operation | list | dict / set | deque |
|---|---|---|---|
| index `a[i]` | O(1) | — | O(n) |
| append / pop at end | O(1)* | — | O(1) |
| insert / pop at front | O(n) | — | O(1) |
| `x in c` | O(n) | O(1) avg | O(n) |
| delete by key | O(n) | O(1) avg | O(n) |
| sort | O(n log n) (Timsort, **stable**) | — | — |
| slice `a[i:j]` | O(j−i) — it **copies** | — | — |

\* amortized

### "What complexity can I afford?" (≈10⁸ simple ops/sec in C, ~10⁷ in Python)

| n up to | Target complexity | Typical technique |
|---|---|---|
| ~10 | O(n!) | permutations, brute force |
| ~20 | O(2ⁿ) | subsets, bitmask DP |
| ~500 | O(n³) | triple loop, interval DP |
| ~5,000 | O(n²) | 2D DP, all pairs |
| ~10⁶ | O(n log n) | sorting, heaps, binary search |
| ≥10⁷ | O(n) or O(log n) | two pointers, hashing, math |

In [ ]:
from collections import deque, defaultdict, Counter, OrderedDict
from functools import cache, lru_cache
from typing import List, Optional
import heapq, bisect, math, itertools, random

def check(name, cond):
    """Tiny test helper so every section prints a green tick."""
    assert cond, f"FAILED: {name}"
    print(f"✓ {name}")

# --- idioms worth having in your fingers ---
a = [5, 2, 9, 1]
check("sorted with key, reverse", sorted(a, key=lambda x: -x) == [9, 5, 2, 1])
check("enumerate", [(i, x) for i, x in enumerate("ab")] == [(0, 'a'), (1, 'b')])
check("zip pairs neighbours", list(zip(a, a[1:])) == [(5, 2), (2, 9), (9, 1)])
check("Counter most_common", Counter("aabbbc").most_common(1) == [('b', 3)])
dd = defaultdict(list); dd["x"].append(1)
check("defaultdict(list)", dd["x"] == [1])
check("2D grid (correct way)", [[0] * 3 for _ in range(2)] == [[0, 0, 0], [0, 0, 0]])
check("divmod", divmod(17, 5) == (3, 2))
check("ceil division without math", -(-7 // 2) == 4)
check("chr/ord", chr(ord('a') + 2) == 'c')
h = []; [heapq.heappush(h, x) for x in a]
check("heapq min at h[0]", h[0] == 1)
check("bisect_left = first index >= x", bisect.bisect_left([1, 3, 3, 5], 3) == 1)
check("bisect_right = first index > x", bisect.bisect_right([1, 3, 3, 5], 3) == 3)

---
# 1. Arrays & Strings

🧠 **Mental model:** A contiguous block of memory. Random access is O(1) because address = base + i·size. Everything clever about arrays is about *avoiding the O(n²) double loop* — usually with **two pointers**, a **sliding window**, or **prefix sums**.

✏️ **Whiteboard sketch**
```
Two pointers (converging)        Sliding window (expanding/shrinking)
 [1, 2, 4, 7, 11, 15]             a b c a b c b b
  L →            ← R                  [L ... R]     R always moves right,
 sum too small → L++                                L moves right only to restore validity
 sum too big   → R--

Prefix sums:  a    =  [3, 1, 4, 1, 5]
              pre  = [0, 3, 4, 8, 9, 14]      sum(a[i:j]) = pre[j] - pre[i]
```

🎯 **Pattern triggers**
- "sorted array", "pair/triplet summing to target" → **two pointers**
- "longest/shortest **contiguous** subarray/substring such that…" → **sliding window**
- "sum of subarray", "number of subarrays with sum k", range-sum queries → **prefix sums (+ hash map)**
- "maximum subarray sum" → **Kadane**

⚠️ **Gotchas:** off-by-one on window length (`r - l + 1`), strings are immutable (build a list and `''.join`), slicing copies.

In [ ]:
# ---------- Two pointers ----------
def two_sum_sorted(nums, target):
    """Indices of two numbers in a SORTED array summing to target. O(n) time, O(1) space."""
    l, r = 0, len(nums) - 1
    while l < r:
        s = nums[l] + nums[r]
        if s == target:
            return l, r
        if s < target:
            l += 1
        else:
            r -= 1
    return None

def three_sum(nums):
    """All unique triplets summing to 0. Sort + fix one + two pointers: O(n^2)."""
    nums = sorted(nums)
    res = []
    for i in range(len(nums) - 2):
        if i and nums[i] == nums[i - 1]:          # skip duplicate anchors
            continue
        l, r = i + 1, len(nums) - 1
        while l < r:
            s = nums[i] + nums[l] + nums[r]
            if s < 0:
                l += 1
            elif s > 0:
                r -= 1
            else:
                res.append([nums[i], nums[l], nums[r]])
                l += 1; r -= 1
                while l < r and nums[l] == nums[l - 1]:   # skip duplicate seconds
                    l += 1
    return res

def max_area(height):
    """Container with most water: move the SHORTER wall inward (it's the bottleneck)."""
    l, r, best = 0, len(height) - 1, 0
    while l < r:
        best = max(best, (r - l) * min(height[l], height[r]))
        if height[l] < height[r]:
            l += 1
        else:
            r -= 1
    return best

def remove_duplicates_sorted(nums):
    """In-place dedupe of a sorted list; returns new length. Slow/fast pointers."""
    if not nums:
        return 0
    w = 1                                   # write pointer
    for r in range(1, len(nums)):           # read pointer
        if nums[r] != nums[w - 1]:
            nums[w] = nums[r]
            w += 1
    return w

check("two_sum_sorted", two_sum_sorted([1, 2, 4, 7, 11, 15], 15) == (2, 4))
check("three_sum", three_sum([-1, 0, 1, 2, -1, -4]) == [[-1, -1, 2], [-1, 0, 1]])
check("max_area", max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]) == 49)
arr = [1, 1, 2, 3, 3, 3, 4]; k = remove_duplicates_sorted(arr)
check("remove_duplicates_sorted", arr[:k] == [1, 2, 3, 4])

In [ ]:
# ---------- Sliding window ----------
def longest_unique_substring(s):
    """Length of longest substring without repeating chars. O(n)."""
    last_seen = {}
    left = best = 0
    for right, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= left:
            left = last_seen[ch] + 1          # jump left past the duplicate
        last_seen[ch] = right
        best = max(best, right - left + 1)
    return best

def max_sum_window(nums, k):
    """Fixed-size window: max sum of any k consecutive elements. O(n)."""
    window = sum(nums[:k])
    best = window
    for i in range(k, len(nums)):
        window += nums[i] - nums[i - k]       # slide: add new, drop old
        best = max(best, window)
    return best

def min_window(s, t):
    """Smallest substring of s containing all chars of t (with multiplicity). O(|s|+|t|).
    Template: expand right until valid, then shrink left while still valid."""
    need = Counter(t)
    missing = len(t)
    left, best = 0, (0, math.inf)
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1                         # chars not in t go negative = "surplus"
        if missing == 0:                      # window valid → shrink
            while need[s[left]] < 0:
                need[s[left]] += 1
                left += 1
            if right - left < best[1] - best[0]:
                best = (left, right)
            need[s[left]] += 1                # give up s[left] to look for the next window
            missing += 1
            left += 1
    return "" if best[1] == math.inf else s[best[0]:best[1] + 1]

check("longest_unique_substring", longest_unique_substring("abcabcbb") == 3)
check("longest_unique_substring edge", longest_unique_substring("") == 0)
check("max_sum_window", max_sum_window([2, 1, 5, 1, 3, 2], 3) == 9)
check("min_window", min_window("ADOBECODEBANC", "ABC") == "BANC")
check("min_window impossible", min_window("a", "aa") == "")

In [ ]:
# ---------- Prefix sums & Kadane ----------
def build_prefix(nums):
    pre = [0]
    for x in nums:
        pre.append(pre[-1] + x)
    return pre                                # range_sum(i, j) = pre[j] - pre[i]  (half-open)

def subarray_sum_equals_k(nums, k):
    """Count subarrays with sum k (works with negatives!). Prefix sum + hash map. O(n).
    Idea: sum(i..j) = pre[j] - pre[i] = k  ⇔  pre[i] = pre[j] - k."""
    seen = Counter({0: 1})
    running = count = 0
    for x in nums:
        running += x
        count += seen[running - k]
        seen[running] += 1
    return count

def kadane(nums):
    """Max subarray sum. At each step: extend the current run or restart here."""
    best = cur = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best

def product_except_self(nums):
    """Prefix products from the left, then suffix products from the right. No division."""
    n = len(nums)
    out = [1] * n
    for i in range(1, n):
        out[i] = out[i - 1] * nums[i - 1]
    suffix = 1
    for i in range(n - 1, -1, -1):
        out[i] *= suffix
        suffix *= nums[i]
    return out

pre = build_prefix([3, 1, 4, 1, 5])
check("prefix range sum", pre[4] - pre[1] == 1 + 4 + 1)
check("subarray_sum_equals_k", subarray_sum_equals_k([1, 1, 1], 2) == 2)
check("subarray_sum_equals_k negatives", subarray_sum_equals_k([3, 4, 7, 2, -3, 1, 4, 2], 7) == 4)
check("kadane", kadane([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6)
check("kadane all negative", kadane([-3, -1, -2]) == -1)
check("product_except_self", product_except_self([1, 2, 3, 4]) == [24, 12, 8, 6])

---
# 2. Hash Maps & Sets

🧠 **Mental model:** An array of buckets. `hash(key) % capacity` picks a bucket; collisions are resolved by chaining or open addressing (CPython uses open addressing). When the table gets too full (~2/3 load), it resizes → amortized O(1). Worst case O(n) if everything collides.

✏️ **Whiteboard sketch**
```
key "apple" ──hash──▶ 0x9f3a… % 8 = 3
buckets: [0][1][2][3: apple→5][4][5][6][7]
```

⏱️ insert / lookup / delete: **O(1) average**, O(n) worst. Keys must be **hashable** (immutable): use `tuple`, not `list`; `frozenset`, not `set`.

🎯 **Pattern triggers**
- "have I seen this before?" / duplicates → `set`
- "count frequencies" / anagrams → `Counter`
- "find complement" (two sum) → `dict` of value → index
- "group by some signature" → `defaultdict(list)` with a canonical key
- Need O(1) lookup to turn an O(n²) scan into O(n)

⚠️ **Gotchas:** iterating a dict while mutating it raises; `Counter` returns 0 for missing keys (dict raises `KeyError`).

In [ ]:
def two_sum(nums, target):
    """Unsorted two sum: store what we've seen; look up the complement. O(n)."""
    seen = {}
    for i, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], i]
        seen[x] = i
    return []

def group_anagrams(words):
    """Canonical key = sorted letters (O(k log k)), or a 26-count tuple (O(k))."""
    groups = defaultdict(list)
    for w in words:
        groups[tuple(sorted(w))].append(w)
    return list(groups.values())

def longest_consecutive(nums):
    """Longest run of consecutive integers in O(n): only start counting at run-starts."""
    s = set(nums)
    best = 0
    for x in s:
        if x - 1 not in s:                 # x starts a run
            y = x
            while y + 1 in s:
                y += 1
            best = max(best, y - x + 1)
    return best

def first_unique_char(s):
    counts = Counter(s)
    for i, ch in enumerate(s):
        if counts[ch] == 1:
            return i
    return -1

check("two_sum", two_sum([2, 7, 11, 15], 9) == [0, 1])
check("group_anagrams", sorted(map(sorted, group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])))
      == [['ate', 'eat', 'tea'], ['bat'], ['nat', 'tan']])
check("longest_consecutive", longest_consecutive([100, 4, 200, 1, 3, 2]) == 4)
check("first_unique_char", first_unique_char("loveleetcode") == 2)

---
# 3. Stacks, Queues & Monotonic Structures

🧠 **Mental model:**
- **Stack (LIFO)** — a pile of plates. Use for *nesting*, *undo*, *"most recent unmatched thing"*, and simulating recursion.
- **Queue (FIFO)** — a line at a counter. Use for *BFS* and *processing in arrival order*.
- **Monotonic stack** — a stack kept sorted; when a new element breaks the order, you pop — and *each pop is an answer* ("the next greater element of the popped item is the new item"). Every element is pushed and popped once → **O(n)**.
- **Monotonic deque** — same idea, but you can also drop from the front when elements leave a window → **sliding-window max/min in O(n)**.

✏️ **Whiteboard sketch**
```
Next greater element for [2, 1, 5, 3]
 i=0 push 2            stack: [2]
 i=1 push 1            stack: [2, 1]       (decreasing)
 i=2 5 > 1 → pop 1, NGE(1)=5
     5 > 2 → pop 2, NGE(2)=5
     push 5            stack: [5]
 i=3 push 3            stack: [5, 3]
 leftovers → NGE = -1
```

⏱️ push/pop O(1). In Python: stack = `list`, queue = `deque` (**never** `list.pop(0)`).

🎯 **Pattern triggers**
- brackets / nested structure / expression evaluation → **stack**
- "next greater/smaller", "days until warmer", "span", histogram area → **monotonic stack**
- "max/min of every window of size k" → **monotonic deque**
- level-by-level / shortest path in unweighted graph → **queue (BFS)**

In [ ]:
def is_valid_parentheses(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = []
    for ch in s:
        if ch in pairs:
            if not stack or stack.pop() != pairs[ch]:
                return False
        else:
            stack.append(ch)
    return not stack                         # leftovers = unmatched openers

def eval_rpn(tokens):
    """Reverse Polish notation. Note int(a / b) truncates toward zero; // floors!"""
    st = []
    for t in tokens:
        if t in "+-*/":
            b, a = st.pop(), st.pop()        # order matters: a op b
            if t == '+': st.append(a + b)
            elif t == '-': st.append(a - b)
            elif t == '*': st.append(a * b)
            else: st.append(int(a / b))
        else:
            st.append(int(t))
    return st[0]

def next_greater(nums):
    res = [-1] * len(nums)
    stack = []                               # indices; values are decreasing
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            res[stack.pop()] = x
        stack.append(i)
    return res

def daily_temperatures(temps):
    """Days to wait for a warmer day. Same monotonic stack; store distance instead of value."""
    res = [0] * len(temps)
    stack = []
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            res[j] = i - j
        stack.append(i)
    return res

def largest_rectangle(heights):
    """Largest rectangle in histogram. Increasing stack; a bar's rectangle ends when a
    shorter bar arrives. Sentinel 0 at the end flushes the stack."""
    stack, best = [], 0                      # stack of indices with increasing heights
    for i, h in enumerate(heights + [0]):
        while stack and heights[stack[-1]] > h:
            height = heights[stack.pop()]
            left = stack[-1] + 1 if stack else 0
            best = max(best, height * (i - left))
        stack.append(i)
    return best

def sliding_window_max(nums, k):
    """Deque holds indices of a DECREASING sequence; front = current max. O(n)."""
    dq, out = deque(), []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:      # smaller items can never be max again
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:                   # front fell out of the window
            dq.popleft()
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out

check("valid parens", is_valid_parentheses("({[]})") and not is_valid_parentheses("(]"))
check("eval_rpn", eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]) == 22)
check("next_greater", next_greater([2, 1, 5, 3]) == [5, 5, -1, -1])
check("daily_temperatures", daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0])
check("largest_rectangle", largest_rectangle([2, 1, 5, 6, 2, 3]) == 10)
check("sliding_window_max", sliding_window_max([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7])

---
# 4. Linked Lists

🧠 **Mental model:** Nodes scattered in memory, each holding a value and a pointer to the next. O(1) insert/delete *if you already hold the node before it*; O(n) to find anything. Almost every linked-list question is solved by one of four tricks:

1. **Dummy head** — removes the "what if the head changes?" special case.
2. **Fast & slow pointers** — middle of list, cycle detection (Floyd), k-th from end.
3. **Reversal** — three pointers: `prev`, `cur`, `nxt`.
4. **Merge** — like merging in merge sort.

✏️ **Whiteboard sketch — reversal (draw this every time!)**
```
prev   cur    nxt
None   [1] → [2] → [3] → None

step:  nxt = cur.next      # save
       cur.next = prev     # flip arrow
       prev = cur          # advance
       cur = nxt

None ← [1]   [2] → [3]      ...end: prev is the new head
```
```
Floyd's cycle:  slow +1, fast +2. If they meet → cycle.
To find the cycle START: reset one pointer to head, move both +1; they meet at the start.
```

⏱️ access O(n), insert/delete at known node O(1), extra space usually O(1).

⚠️ **Gotchas:** losing the rest of the list (save `next` first!), `None.next` errors — check `fast and fast.next`.

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

def build_list(vals):
    dummy = tail = ListNode()
    for v in vals:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def to_list(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

def reverse_list(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev, cur = cur, nxt
    return prev

def reverse_list_recursive(head):
    if not head or not head.next:
        return head
    new_head = reverse_list_recursive(head.next)
    head.next.next = head                    # the node after me now points back to me
    head.next = None
    return new_head

def middle_node(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    return slow                              # second middle for even length

def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

def cycle_start(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            slow = head
            while slow is not fast:
                slow, fast = slow.next, fast.next
            return slow
    return None

def merge_two_sorted(a, b):
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a or b
    return dummy.next

def remove_nth_from_end(head, n):
    """Gap of n between fast and slow; dummy handles removing the head."""
    dummy = ListNode(0, head)
    fast = slow = dummy
    for _ in range(n + 1):
        fast = fast.next
    while fast:
        fast, slow = fast.next, slow.next
    slow.next = slow.next.next
    return dummy.next

check("reverse_list", to_list(reverse_list(build_list([1, 2, 3, 4]))) == [4, 3, 2, 1])
check("reverse_list_recursive", to_list(reverse_list_recursive(build_list([1, 2, 3]))) == [3, 2, 1])
check("middle_node", middle_node(build_list([1, 2, 3, 4, 5])).val == 3)
cyc = build_list([3, 2, 0, -4]); cyc.next.next.next.next = cyc.next
check("has_cycle", has_cycle(cyc) and not has_cycle(build_list([1, 2])))
check("cycle_start", cycle_start(cyc).val == 2)
check("merge_two_sorted", to_list(merge_two_sorted(build_list([1, 2, 4]), build_list([1, 3, 4]))) == [1, 1, 2, 3, 4, 4])
check("remove_nth_from_end", to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5])
check("remove head", to_list(remove_nth_from_end(build_list([1]), 1)) == [])

---
# 5. Trees & Binary Search Trees

🧠 **Mental model:** A tree is a recursive structure: *a node plus some subtrees*. So most tree solutions are: **"solve for left, solve for right, combine at the root."** Ask yourself: *what does each recursive call return?*

- **Binary tree:** ≤ 2 children. Height h; balanced → h = O(log n), skewed → h = O(n).
- **BST invariant:** everything in left subtree < node < everything in right subtree. ⇒ **in-order traversal yields sorted order.**

✏️ **Whiteboard sketch**
```
        4             Pre-order  (root, L, R): 4 2 1 3 6 5 7   ← copy/serialize a tree
      /   \           In-order   (L, root, R): 1 2 3 4 5 6 7   ← BST → sorted!
     2     6          Post-order (L, R, root): 1 3 2 5 7 6 4   ← delete/compute from children up
    / \   / \         Level-order (BFS):       4 | 2 6 | 1 3 5 7
   1   3 5   7
```

⏱️ Traversals O(n) time, O(h) space (recursion stack). BST search/insert/delete O(h): O(log n) balanced, O(n) worst.

🎯 **Pattern triggers**
- "depth", "diameter", "balanced", "path sum" → **DFS returning a value up**
- "level by level", "right side view", "min depth" → **BFS with a queue**
- "kth smallest in BST", "validate BST" → **in-order** / bounds
- "lowest common ancestor" → recursion that returns "found something"

⚠️ **Gotchas:** validating a BST by only checking `left < node < right` locally is **wrong** — you need min/max bounds down the path. Python recursion limit ≈ 1000 → for deep (skewed) trees use iterative versions.

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build_tree(vals):
    """Build from LeetCode-style level-order list, e.g. [4,2,6,1,3,None,7]."""
    if not vals or vals[0] is None:
        return None
    it = iter(vals)
    root = TreeNode(next(it))
    q = deque([root])
    while q:
        node = q.popleft()
        for side in ("left", "right"):
            try:
                v = next(it)
            except StopIteration:
                return root
            if v is not None:
                child = TreeNode(v)
                setattr(node, side, child)
                q.append(child)
    return root

# ---- Recursive traversals: 3 lines each ----
def preorder(n):  return [n.val] + preorder(n.left) + preorder(n.right) if n else []
def inorder(n):   return inorder(n.left) + [n.val] + inorder(n.right) if n else []
def postorder(n): return postorder(n.left) + postorder(n.right) + [n.val] if n else []

# ---- Iterative traversals (interviewers love asking for these) ----
def inorder_iter(root):
    res, st, cur = [], [], root
    while cur or st:
        while cur:                 # go as far left as possible
            st.append(cur)
            cur = cur.left
        cur = st.pop()             # visit
        res.append(cur.val)
        cur = cur.right            # then explore right subtree
    return res

def preorder_iter(root):
    if not root: return []
    res, st = [], [root]
    while st:
        n = st.pop()
        res.append(n.val)
        if n.right: st.append(n.right)   # push right first so left is processed first
        if n.left:  st.append(n.left)
    return res

def postorder_iter(root):
    """Trick: do (root, right, left) then reverse → (left, right, root)."""
    if not root: return []
    res, st = [], [root]
    while st:
        n = st.pop()
        res.append(n.val)
        if n.left:  st.append(n.left)
        if n.right: st.append(n.right)
    return res[::-1]

def level_order(root):
    if not root: return []
    res, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):          # freeze the level size!
            n = q.popleft()
            level.append(n.val)
            if n.left:  q.append(n.left)
            if n.right: q.append(n.right)
        res.append(level)
    return res

t = build_tree([4, 2, 6, 1, 3, 5, 7])
check("preorder", preorder(t) == preorder_iter(t) == [4, 2, 1, 3, 6, 5, 7])
check("inorder", inorder(t) == inorder_iter(t) == [1, 2, 3, 4, 5, 6, 7])
check("postorder", postorder(t) == postorder_iter(t) == [1, 3, 2, 5, 7, 6, 4])
check("level_order", level_order(t) == [[4], [2, 6], [1, 3, 5, 7]])

In [ ]:
# ---- Classic "return something upward" problems ----
def max_depth(root):
    return 1 + max(max_depth(root.left), max_depth(root.right)) if root else 0

def diameter(root):
    """Longest path (in edges) between any two nodes. Return height, update global best."""
    best = 0
    def height(n):
        nonlocal best
        if not n: return 0
        l, r = height(n.left), height(n.right)
        best = max(best, l + r)           # path bending through n
        return 1 + max(l, r)
    height(root)
    return best

def is_balanced(root):
    """Return -1 as a 'poison' value meaning unbalanced, else height. O(n)."""
    def h(n):
        if not n: return 0
        l, r = h(n.left), h(n.right)
        if l < 0 or r < 0 or abs(l - r) > 1:
            return -1
        return 1 + max(l, r)
    return h(root) >= 0

def is_valid_bst(root, lo=-math.inf, hi=math.inf):
    if not root: return True
    if not (lo < root.val < hi): return False
    return is_valid_bst(root.left, lo, root.val) and is_valid_bst(root.right, root.val, hi)

def has_path_sum(root, target):
    if not root: return False
    if not root.left and not root.right:
        return root.val == target
    rest = target - root.val
    return has_path_sum(root.left, rest) or has_path_sum(root.right, rest)

def invert(root):
    if root:
        root.left, root.right = invert(root.right), invert(root.left)
    return root

def lca(root, p, q):
    """LCA in a general binary tree: return the node if found in this subtree."""
    if not root or root.val in (p, q):
        return root
    l, r = lca(root.left, p, q), lca(root.right, p, q)
    if l and r: return root               # p and q split here
    return l or r

def lca_bst(root, p, q):
    """In a BST just walk: both smaller → go left; both bigger → go right; else here."""
    while root:
        if p < root.val and q < root.val: root = root.left
        elif p > root.val and q > root.val: root = root.right
        else: return root

def kth_smallest(root, k):
    st, cur = [], root
    while True:
        while cur:
            st.append(cur); cur = cur.left
        cur = st.pop()
        k -= 1
        if k == 0: return cur.val
        cur = cur.right

check("max_depth", max_depth(t) == 3)
check("diameter", diameter(build_tree([1, 2, 3, 4, 5])) == 3)
check("is_balanced", is_balanced(t) and not is_balanced(build_tree([1, 2, None, 3])))
check("is_valid_bst", is_valid_bst(t) and not is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])))
check("has_path_sum", has_path_sum(t, 4 + 6 + 7))
check("lca", lca(t, 1, 3).val == 2 and lca(t, 1, 7).val == 4)
check("lca_bst", lca_bst(t, 5, 7).val == 6)
check("kth_smallest", kth_smallest(t, 3) == 3)
check("invert", level_order(invert(build_tree([4, 2, 6]))) == [[4], [6, 2]])

In [ ]:
# ---- BST operations: search / insert / delete ----
def bst_search(root, x):
    while root and root.val != x:
        root = root.left if x < root.val else root.right
    return root

def bst_insert(root, x):
    if not root: return TreeNode(x)
    if x < root.val: root.left = bst_insert(root.left, x)
    else:            root.right = bst_insert(root.right, x)
    return root

def bst_delete(root, key):
    """Three cases: leaf → remove; one child → splice; two children → copy in-order
    successor (min of right subtree), then delete the successor."""
    if not root: return None
    if key < root.val:
        root.left = bst_delete(root.left, key)
    elif key > root.val:
        root.right = bst_delete(root.right, key)
    else:
        if not root.left:  return root.right
        if not root.right: return root.left
        succ = root.right
        while succ.left:
            succ = succ.left
        root.val = succ.val
        root.right = bst_delete(root.right, succ.val)
    return root

def serialize(root):
    """Pre-order with '#' for None → string. Deserialize consumes the same order."""
    out = []
    def go(n):
        if not n: out.append("#"); return
        out.append(str(n.val)); go(n.left); go(n.right)
    go(root)
    return ",".join(out)

def deserialize(data):
    vals = iter(data.split(","))
    def go():
        v = next(vals)
        if v == "#": return None
        n = TreeNode(int(v))
        n.left, n.right = go(), go()
        return n
    return go()

root = None
for x in [50, 30, 70, 20, 40, 60, 80]:
    root = bst_insert(root, x)
check("bst insert/inorder", inorder(root) == [20, 30, 40, 50, 60, 70, 80])
check("bst_search", bst_search(root, 60).val == 60 and bst_search(root, 65) is None)
root = bst_delete(root, 50)   # two children
root = bst_delete(root, 20)   # leaf
check("bst_delete", inorder(root) == [30, 40, 60, 70, 80] and is_valid_bst(root))
check("serialize round trip", inorder(deserialize(serialize(t))) == inorder(t))

---
# 6. Heaps / Priority Queues

🧠 **Mental model:** A **complete binary tree stored in an array** where every parent ≤ its children (min-heap). You only ever know the *minimum* — and that's all you need for "give me the smallest/largest next".

✏️ **Whiteboard sketch**
```
array:  [1, 3, 2, 7, 4, 5]          index math (0-based):
             1                        parent(i) = (i - 1) // 2
           /   \                      left(i)   = 2i + 1
          3     2                     right(i)  = 2i + 2
         / \   /
        7   4 5
push → append at end, SIFT UP (swap with parent while smaller)
pop  → move last to root, SIFT DOWN (swap with smaller child while bigger)
```

⏱️ push O(log n), pop O(log n), peek O(1), **heapify O(n)** (not n log n — say this, it impresses).

🎯 **Pattern triggers**
- "top K", "K-th largest/smallest", "K closest" → heap of size K
- "merge K sorted …" → heap of K heads
- "running median" → two heaps
- "schedule by earliest end / priority" → heap
- Dijkstra / best-first search

⚠️ **Gotchas (Python):** `heapq` is **min-heap only** → push `-x` for a max-heap. When pushing tuples, ties compare the next element — if that's an object without `<`, it crashes → insert a counter: `(priority, count, obj)`.

In [ ]:
class MinHeap:
    """From scratch — be able to write sift_up / sift_down on a whiteboard."""
    def __init__(self, items=None):
        self.a = list(items or [])
        for i in reversed(range(len(self.a) // 2)):   # O(n) heapify: sift down non-leaves
            self._sift_down(i)

    def __len__(self): return len(self.a)
    def peek(self): return self.a[0]

    def push(self, x):
        self.a.append(x)
        self._sift_up(len(self.a) - 1)

    def pop(self):
        a = self.a
        if not a: raise IndexError("pop from empty heap")
        top, last = a[0], a.pop()
        if a:
            a[0] = last
            self._sift_down(0)
        return top

    def _sift_up(self, i):
        a = self.a
        while i > 0:
            p = (i - 1) // 2
            if a[i] >= a[p]: break
            a[i], a[p] = a[p], a[i]
            i = p

    def _sift_down(self, i):
        a, n = self.a, len(self.a)
        while True:
            l, r, smallest = 2 * i + 1, 2 * i + 2, i
            if l < n and a[l] < a[smallest]: smallest = l
            if r < n and a[r] < a[smallest]: smallest = r
            if smallest == i: return
            a[i], a[smallest] = a[smallest], a[i]
            i = smallest

data = [random.randint(0, 100) for _ in range(50)]
h = MinHeap(data[:25])
for x in data[25:]: h.push(x)
check("MinHeap sorts", [h.pop() for _ in range(len(h))] == sorted(data))

In [ ]:
def kth_largest(nums, k):
    """Keep a MIN-heap of the k largest seen so far; its root is the answer. O(n log k)."""
    h = []
    for x in nums:
        heapq.heappush(h, x)
        if len(h) > k:
            heapq.heappop(h)
    return h[0]

def top_k_frequent(nums, k):
    counts = Counter(nums)
    return [x for x, _ in heapq.nlargest(k, counts.items(), key=lambda kv: kv[1])]

def k_closest_points(points, k):
    """Max-heap (negated distances) of size k."""
    h = []
    for x, y in points:
        heapq.heappush(h, (-(x * x + y * y), x, y))
        if len(h) > k:
            heapq.heappop(h)
    return sorted([[x, y] for _, x, y in h])

def merge_k_sorted_lists(lists):
    """Heap of (value, tiebreak index, node). O(N log k)."""
    h = [(node.val, i, node) for i, node in enumerate(lists) if node]
    heapq.heapify(h)
    dummy = tail = ListNode()
    while h:
        val, i, node = heapq.heappop(h)
        tail.next = node
        tail = node
        if node.next:
            heapq.heappush(h, (node.next.val, i, node.next))
    return dummy.next

class MedianFinder:
    """Two heaps: `lo` = max-heap of the smaller half (negated), `hi` = min-heap of the
    larger half. Invariant: len(lo) == len(hi) or len(lo) == len(hi) + 1."""
    def __init__(self):
        self.lo, self.hi = [], []

    def add(self, x):
        heapq.heappush(self.lo, -x)
        heapq.heappush(self.hi, -heapq.heappop(self.lo))   # move lo's max to hi
        if len(self.hi) > len(self.lo):                     # rebalance
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def median(self):
        if len(self.lo) > len(self.hi):
            return -self.lo[0]
        return (-self.lo[0] + self.hi[0]) / 2

check("kth_largest", kth_largest([3, 2, 3, 1, 2, 4, 5, 5, 6], 4) == 4)
check("top_k_frequent", sorted(top_k_frequent([1, 1, 1, 2, 2, 3], 2)) == [1, 2])
check("k_closest_points", k_closest_points([[3, 3], [5, -1], [-2, 4]], 2) == [[-2, 4], [3, 3]])
merged = merge_k_sorted_lists([build_list([1, 4, 5]), build_list([1, 3, 4]), build_list([2, 6])])
check("merge_k_sorted_lists", to_list(merged) == [1, 1, 2, 3, 4, 4, 5, 6])
mf = MedianFinder(); meds = []
for x in [5, 15, 1, 3]:
    mf.add(x); meds.append(mf.median())
check("MedianFinder", meds == [5, 10.0, 5, 4.0])

---
# 7. Binary Search

🧠 **Mental model:** Binary search isn't about sorted arrays — it's about a **monotonic predicate**: a yes/no question whose answers look like `F F F F T T T`. You're finding the *first T*. Once you see it that way, you can binary search on indices, on rotated arrays, and even on **the answer itself** ("what's the minimum speed/capacity such that it's feasible?").

✏️ **Whiteboard sketch — the one template to memorize**
```
lo, hi = 0, n            # search space [lo, hi)  — hi is "past the end"
while lo < hi:
    mid = (lo + hi) // 2
    if pred(mid): hi = mid       # mid might be the answer, keep it
    else:         lo = mid + 1   # mid is definitely not
return lo                # first index where pred is True (n if none)

 F  F  F  T  T  T
          ^ lo == hi at the end
```

⏱️ O(log n) iterations × cost of the predicate.

🎯 **Pattern triggers**
- sorted input + "find / first / last / insert position"
- "minimum X such that…" / "maximum X such that…" where feasibility is monotonic → **binary search on the answer**
- rotated sorted array, peak element, "log n required"

⚠️ **Gotchas:** infinite loops from `lo = mid` with `lo < hi` (use `mid + 1`), mixing inclusive/exclusive bounds. Pick one template and stick with it.

In [ ]:
def first_true(lo, hi, pred):
    """Smallest x in [lo, hi) with pred(x) True; returns hi if none."""
    while lo < hi:
        mid = (lo + hi) // 2
        if pred(mid): hi = mid
        else:         lo = mid + 1
    return lo

def lower_bound(a, x):   return first_true(0, len(a), lambda i: a[i] >= x)
def upper_bound(a, x):   return first_true(0, len(a), lambda i: a[i] > x)

def search_range(a, x):
    """First and last position of x (LeetCode 34)."""
    l = lower_bound(a, x)
    if l == len(a) or a[l] != x:
        return [-1, -1]
    return [l, upper_bound(a, x) - 1]

def search_rotated(nums, target):
    """One half of [lo, hi] is always sorted — check whether target lies in it."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target: return mid
        if nums[lo] <= nums[mid]:                       # left half sorted
            if nums[lo] <= target < nums[mid]: hi = mid - 1
            else: lo = mid + 1
        else:                                           # right half sorted
            if nums[mid] < target <= nums[hi]: lo = mid + 1
            else: hi = mid - 1
    return -1

def find_min_rotated(nums):
    """Predicate: nums[i] <= nums[-1]  →  F F F T T T; the first T is the min."""
    return nums[first_true(0, len(nums), lambda i: nums[i] <= nums[-1])]

def find_peak(nums):
    """Any peak: predicate nums[i] > nums[i+1] is 'eventually true'."""
    return first_true(0, len(nums) - 1, lambda i: nums[i] > nums[i + 1])

def int_sqrt(x):
    """Largest r with r*r <= x  ==  (first r with r*r > x) - 1."""
    return first_true(0, x + 2, lambda r: r * r > x) - 1

def min_eating_speed(piles, h):
    """Koko bananas — binary search on the ANSWER. Feasible(k) is monotonic in k."""
    feasible = lambda k: sum(-(-p // k) for p in piles) <= h
    return first_true(1, max(piles) + 1, feasible)

def ship_within_days(weights, days):
    """Min ship capacity to ship in order within `days`. Same pattern."""
    def feasible(cap):
        need, cur = 1, 0
        for w in weights:
            if cur + w > cap:
                need, cur = need + 1, 0
            cur += w
        return need <= days
    return first_true(max(weights), sum(weights) + 1, feasible)

a = [1, 2, 2, 2, 5, 7]
check("lower/upper bound", (lower_bound(a, 2), upper_bound(a, 2)) == (1, 4))
check("matches bisect", lower_bound(a, 3) == bisect.bisect_left(a, 3))
check("search_range", search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4])
check("search_rotated", search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4 and search_rotated([4, 5, 6, 7, 0, 1, 2], 3) == -1)
check("find_min_rotated", find_min_rotated([4, 5, 6, 7, 0, 1, 2]) == 0)
check("find_peak", find_peak([1, 2, 3, 1]) == 2)
check("int_sqrt", [int_sqrt(x) for x in [0, 1, 8, 9, 10]] == [0, 1, 2, 3, 3])
check("min_eating_speed", min_eating_speed([3, 6, 7, 11], 8) == 4)
check("ship_within_days", ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5) == 15)

---
# 8. Graphs

🧠 **Mental model:** Vertices + edges. Trees, grids, dependency lists, state machines, currency-exchange tables — all graphs. The first question is always **"what are the nodes and what are the edges?"**, the second is **"weighted or unweighted? directed or not?"**

| Problem | Tool | Cost |
|---|---|---|
| reachability / connected components | DFS or BFS or Union-Find | O(V + E) |
| shortest path, **unweighted** | BFS | O(V + E) |
| shortest path, **non-negative weights** | Dijkstra (heap) | O((V + E) log V) |
| shortest path with negative weights | Bellman-Ford | O(V·E) |
| ordering with dependencies, cycle in DAG | Topological sort (Kahn / DFS) | O(V + E) |
| dynamic connectivity, "are these connected", MST (Kruskal) | Union-Find | ~O(α(n)) per op |

✏️ **Whiteboard sketch — representations**
```
   0 ── 1              adjacency list (default!)      adjacency matrix
   │  ╱ │              {0: [1, 2],                    [[0,1,1,0],
   2    3               1: [0, 2, 3],                  [1,0,1,1],
                        2: [0, 1],                     [1,1,0,0],
                        3: [1]}                        [0,1,0,0]]
                       space O(V + E)                 space O(V²), O(1) edge check
Grid: node = (r, c), edges = 4 neighbours: for dr, dc in ((1,0),(-1,0),(0,1),(0,-1))
```

⚠️ **Gotchas:** mark visited **when you enqueue** (not when you pop) in BFS, or you'll enqueue duplicates. Recursive DFS on big grids can hit the recursion limit → iterative stack.

In [ ]:
DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))

def bfs_shortest_path(graph, src, dst):
    """Unweighted shortest path; returns the path itself via a parent map."""
    parent = {src: None}
    q = deque([src])
    while q:
        u = q.popleft()
        if u == dst:
            path = []
            while u is not None:
                path.append(u); u = parent[u]
            return path[::-1]
        for v in graph.get(u, []):
            if v not in parent:              # visited-on-enqueue
                parent[v] = u
                q.append(v)
    return None

def dfs_iterative(graph, src):
    seen, order, st = set(), [], [src]
    while st:
        u = st.pop()
        if u in seen: continue
        seen.add(u); order.append(u)
        for v in reversed(graph.get(u, [])):   # reversed → same order as recursive
            if v not in seen:
                st.append(v)
    return order

def num_islands(grid):
    """Count connected '1' regions. BFS flood-fill each new island."""
    grid = [row[:] for row in grid]
    R, C, count = len(grid), len(grid[0]), 0
    for r in range(R):
        for c in range(C):
            if grid[r][c] == "1":
                count += 1
                grid[r][c] = "0"
                q = deque([(r, c)])
                while q:
                    i, j = q.popleft()
                    for di, dj in DIRS:
                        ni, nj = i + di, j + dj
                        if 0 <= ni < R and 0 <= nj < C and grid[ni][nj] == "1":
                            grid[ni][nj] = "0"
                            q.append((ni, nj))
    return count

def rotting_oranges(grid):
    """Multi-source BFS: start the queue with ALL sources at time 0."""
    R, C = len(grid), len(grid[0])
    q = deque((r, c) for r in range(R) for c in range(C) if grid[r][c] == 2)
    fresh = sum(row.count(1) for row in grid)
    grid = [row[:] for row in grid]
    minutes = 0
    while q and fresh:
        minutes += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] == 1:
                    grid[nr][nc] = 2; fresh -= 1; q.append((nr, nc))
    return minutes if fresh == 0 else -1

g = {0: [1, 2], 1: [0, 2, 3], 2: [0, 1], 3: [1]}
check("bfs_shortest_path", bfs_shortest_path(g, 0, 3) == [0, 1, 3])
check("dfs_iterative", dfs_iterative(g, 0) == [0, 1, 2, 3])
check("num_islands", num_islands([list("11000"), list("11000"), list("00100"), list("00011")]) == 3)
check("rotting_oranges", rotting_oranges([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4)

In [ ]:
def topo_sort_kahn(n, edges):
    """Kahn's algorithm: repeatedly take nodes with in-degree 0.
    If we can't take all n, there's a cycle → return []."""
    g = defaultdict(list)
    indeg = [0] * n
    for u, v in edges:                     # u must come before v
        g[u].append(v); indeg[v] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order if len(order) == n else []

def has_cycle_directed(n, edges):
    """DFS with 3 colours: 0 = unvisited, 1 = on current path, 2 = done.
    Edge into a node coloured 1 = back edge = cycle."""
    g = defaultdict(list)
    for u, v in edges: g[u].append(v)
    color = [0] * n
    def dfs(u):
        color[u] = 1
        for v in g[u]:
            if color[v] == 1 or (color[v] == 0 and dfs(v)):
                return True
        color[u] = 2
        return False
    return any(color[i] == 0 and dfs(i) for i in range(n))

def dijkstra(graph, src):
    """graph: {u: [(v, w), ...]}, w >= 0. Lazy deletion: skip stale heap entries."""
    dist = {src: 0}
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:                    # stale entry
            continue
        for v, w in graph.get(u, []):
            nd = d + w
            if nd < dist.get(v, math.inf):
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

def bellman_ford(n, edges, src):
    """Handles negative weights; detects negative cycles (→ None)."""
    dist = [math.inf] * n; dist[src] = 0
    for _ in range(n - 1):
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    for u, v, w in edges:
        if dist[u] + w < dist[v]:
            return None
    return dist

check("topo_sort_kahn", topo_sort_kahn(4, [(0, 1), (0, 2), (1, 3), (2, 3)]) in ([0, 1, 2, 3], [0, 2, 1, 3]))
check("topo detects cycle", topo_sort_kahn(2, [(0, 1), (1, 0)]) == [])
check("has_cycle_directed", has_cycle_directed(3, [(0, 1), (1, 2), (2, 0)]) and not has_cycle_directed(3, [(0, 1), (1, 2)]))
wg = {'A': [('B', 1), ('C', 4)], 'B': [('C', 2), ('D', 5)], 'C': [('D', 1)]}
check("dijkstra", dijkstra(wg, 'A') == {'A': 0, 'B': 1, 'C': 3, 'D': 4})
check("bellman_ford", bellman_ford(3, [(0, 1, 4), (0, 2, 5), (1, 2, -3)], 0) == [0, 4, 1])
check("bellman_ford neg cycle", bellman_ford(2, [(0, 1, 1), (1, 0, -2)], 0) is None)

### Union-Find (Disjoint Set Union)

🧠 **Mental model:** Every element points to a parent; following parents leads to the set's **representative (root)**. Two elements are in the same set iff they share a root.
- **Path compression:** while finding, point nodes closer to the root.
- **Union by size/rank:** hang the smaller tree under the bigger one.
With both, operations are effectively O(1) (inverse Ackermann).

```
union(1,2), union(3,4), union(2,4):
   1          1
   │   →    ╱   ╲
   2       2     3
                 │
                 4
```

🎯 Triggers: "number of connected components", "redundant connection", "accounts merge", "are these in the same group" with edges arriving online, Kruskal's MST.

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.components = n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path halving
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                                    # already connected (cycle!)
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.components -= 1
        return True

def redundant_connection(edges):
    """First edge that creates a cycle in an undirected graph."""
    dsu = DSU(len(edges) + 1)
    for u, v in edges:
        if not dsu.union(u, v):
            return [u, v]

def kruskal_mst(n, edges):
    """edges: (w, u, v). Sort by weight, add if it joins two components."""
    dsu, total = DSU(n), 0
    for w, u, v in sorted(edges):
        if dsu.union(u, v):
            total += w
    return total

d = DSU(5); d.union(0, 1); d.union(3, 4)
check("DSU components", d.components == 3 and d.find(0) == d.find(1) and d.find(1) != d.find(3))
check("redundant_connection", redundant_connection([[1, 2], [1, 3], [2, 3]]) == [2, 3])
check("kruskal_mst", kruskal_mst(4, [(1, 0, 1), (4, 0, 2), (3, 1, 2), (2, 1, 3), (5, 2, 3)]) == 6)

---
# 9. Tries (Prefix Trees)

🧠 **Mental model:** A tree where each edge is a character; a path from the root spells a prefix. Nodes mark whether a full word ends there. Lookup cost depends on **word length L**, not on how many words are stored.

✏️ **Whiteboard sketch** for `{"car", "cat", "cart", "dog"}`
```
        (root)
        /    \
       c      d
       |      |
       a      o
      / \     |
     r*  t*   g*        * = end of word
     |
     t*
```

⏱️ insert/search/startsWith O(L). Space O(total characters).

🎯 Triggers: autocomplete, "starts with", word search on a board with many words, longest common prefix, IP routing (binary trie).

In [ ]:
class TrieNode:
    __slots__ = ("children", "end")
    def __init__(self):
        self.children = {}
        self.end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.end = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def search(self, word):
        node = self._walk(word)
        return bool(node and node.end)

    def starts_with(self, prefix):
        return self._walk(prefix) is not None

    def autocomplete(self, prefix, limit=10):
        node, out = self._walk(prefix), []
        def dfs(n, path):
            if len(out) >= limit: return
            if n.end: out.append(prefix + path)
            for ch in sorted(n.children):
                dfs(n.children[ch], path + ch)
        if node: dfs(node, "")
        return out

tr = Trie()
for w in ["car", "cat", "cart", "dog"]:
    tr.insert(w)
check("trie search", tr.search("car") and not tr.search("ca"))
check("trie starts_with", tr.starts_with("ca") and not tr.starts_with("cb"))
check("trie autocomplete", tr.autocomplete("ca") == ["car", "cart", "cat"])

---
# 10. Dynamic Programming

🧠 **Mental model:** DP = **recursion + memory**. If a brute-force recursion solves the same subproblem many times, store the answers. Use this 4-step recipe out loud on the whiteboard:

1. **State** — what does `dp[i]` (or `dp[i][j]`) *mean*, in one English sentence?
2. **Transition** — how is `dp[i]` built from smaller states? (the "last decision" trick: think about the last element/step)
3. **Base cases** — the smallest states.
4. **Order & answer** — fill order (so dependencies are ready) and which cell holds the answer.

Then optionally: **space optimize** (if `dp[i]` only needs `dp[i-1]`, keep two variables / one row).

✏️ **Whiteboard sketch — LCS table for "abcde" vs "ace"**
```
        ""  a  c  e
    ""   0  0  0  0
    a    0  1  1  1        match:    dp[i][j] = dp[i-1][j-1] + 1      (↖ + 1)
    b    0  1  1  1        no match: dp[i][j] = max(dp[i-1][j], dp[i][j-1])  (↑ or ←)
    c    0  1  2  2
    d    0  1  2  2
    e    0  1  2  3   ← answer
```

| Family | Classic examples | State shape |
|---|---|---|
| 1-D linear | climbing stairs, house robber, decode ways | `dp[i]` = best for prefix of length i |
| Unbounded choice | coin change | `dp[amount]` |
| 0/1 knapsack | subset sum, partition equal subset | `dp[i][cap]` → 1-D reversed |
| Two sequences | LCS, edit distance | `dp[i][j]` over prefixes |
| Grid | unique paths, min path sum | `dp[r][c]` |
| State machine | stock buy/sell with cooldown/fee | a few variables per day |
| LIS | longest increasing subsequence | `dp[i]` or patience-sorting tails |

⚠️ **Gotchas:** top-down `@cache` is fastest to write in an interview — say "let me write the memoized recursion first, then convert to bottom-up if you'd like". Watch the recursion limit for n ≳ 1000.

In [ ]:
# ---- Top-down vs bottom-up on the simplest example ----
@cache
def fib_memo(n):
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

def fib_tab(n):
    a, b = 0, 1                      # O(1) space
    for _ in range(n):
        a, b = b, a + b
    return a

def climb_stairs(n):
    """Ways to climb n steps taking 1 or 2: dp[i] = dp[i-1] + dp[i-2] (fib in disguise)."""
    a, b = 1, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

def house_robber(nums):
    """dp[i] = max(skip i: dp[i-1], take i: dp[i-2] + nums[i])."""
    prev2 = prev1 = 0
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1

def coin_change(coins, amount):
    """Fewest coins. dp[a] = 1 + min(dp[a - c]). Unbounded → iterate amounts outer."""
    dp = [0] + [math.inf] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] != math.inf else -1

def coin_change_ways(coins, amount):
    """Number of COMBINATIONS. Coins outer loop → each combination counted once."""
    dp = [1] + [0] * amount
    for c in coins:
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

def decode_ways(s):
    """'12' → 'AB' or 'L'. dp[i] = ways to decode s[:i]."""
    prev2, prev1 = 1, 1 if s and s[0] != '0' else 0
    for i in range(1, len(s)):
        cur = prev1 if s[i] != '0' else 0
        if 10 <= int(s[i - 1:i + 1]) <= 26:
            cur += prev2
        prev2, prev1 = prev1, cur
    return prev1 if s else 0

check("fib", fib_memo(30) == fib_tab(30) == 832040)
check("climb_stairs", climb_stairs(5) == 8)
check("house_robber", house_robber([2, 7, 9, 3, 1]) == 12)
check("coin_change", coin_change([1, 2, 5], 11) == 3 and coin_change([2], 3) == -1)
check("coin_change_ways", coin_change_ways([1, 2, 5], 5) == 4)
check("decode_ways", decode_ways("226") == 3 and decode_ways("06") == 0)

In [ ]:
# ---- Sequences, grids, knapsack ----
def lis_n2(nums):
    """dp[i] = length of LIS ending at i. O(n^2) — explain this first."""
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp, default=0)

def lis_nlogn(nums):
    """tails[k] = smallest possible tail of an increasing subsequence of length k+1.
    tails stays sorted → binary search where each x goes. O(n log n)."""
    tails = []
    for x in nums:
        i = bisect.bisect_left(tails, x)
        if i == len(tails): tails.append(x)
        else: tails[i] = x
    return len(tails)

def lcs(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

def edit_distance(a, b):
    """dp[i][j] = min edits to turn a[:i] into b[:j]. Insert ←, delete ↑, replace ↖."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n]

def unique_paths(m, n):
    row = [1] * n                    # 1-row space optimization
    for _ in range(1, m):
        for j in range(1, n):
            row[j] += row[j - 1]
    return row[-1]

def min_path_sum(grid):
    R, C = len(grid), len(grid[0])
    dp = [math.inf] * (C + 1); dp[1] = 0
    for r in range(R):
        for c in range(C):
            dp[c + 1] = grid[r][c] + min(dp[c], dp[c + 1])   # left, up
    return dp[C]

def knapsack_01(weights, values, cap):
    """Each item once. 1-D dp iterated BACKWARDS so an item isn't reused."""
    dp = [0] * (cap + 1)
    for w, v in zip(weights, values):
        for c in range(cap, w - 1, -1):
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[cap]

def can_partition(nums):
    """Equal subset sum = 0/1 knapsack on booleans."""
    total = sum(nums)
    if total % 2: return False
    target = total // 2
    reach = {0}
    for x in nums:
        reach |= {s + x for s in reach if s + x <= target}
    return target in reach

def word_break(s, words):
    ws = set(words)
    dp = [True] + [False] * len(s)   # dp[i]: s[:i] can be segmented
    for i in range(1, len(s) + 1):
        dp[i] = any(dp[j] and s[j:i] in ws for j in range(i))
    return dp[-1]

check("lis", lis_n2([10, 9, 2, 5, 3, 7, 101, 18]) == lis_nlogn([10, 9, 2, 5, 3, 7, 101, 18]) == 4)
check("lcs", lcs("abcde", "ace") == 3)
check("edit_distance", edit_distance("horse", "ros") == 3)
check("unique_paths", unique_paths(3, 7) == 28)
check("min_path_sum", min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7)
check("knapsack_01", knapsack_01([1, 3, 4, 5], [1, 4, 5, 7], 7) == 9)
check("can_partition", can_partition([1, 5, 11, 5]) and not can_partition([1, 2, 3, 5]))
check("word_break", word_break("leetcode", ["leet", "code"]) and not word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]))

### 📈 Stock problems = state-machine DP (quant interview favourites)

Think of each day as a transition between states. Draw this:
```
            buy (-p)
   [rest] ───────────▶ [hold]
     ▲  ↺               ↺  │
     │                     │ sell (+p)
     └──── cooldown ── [sold] ◀┘
```
Each state stores "best cash if I end today in this state". Update all states from yesterday's values simultaneously.

In [ ]:
def max_profit_one(prices):
    """One transaction: track the cheapest price so far."""
    lo, best = math.inf, 0
    for p in prices:
        lo = min(lo, p)
        best = max(best, p - lo)
    return best

def max_profit_unlimited(prices):
    """Unlimited transactions: sum every upward move."""
    return sum(max(0, b - a) for a, b in zip(prices, prices[1:]))

def max_profit_cooldown(prices):
    hold, sold, rest = -math.inf, 0, 0
    for p in prices:
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return max(sold, rest)

def max_profit_fee(prices, fee):
    cash, hold = 0, -math.inf
    for p in prices:
        cash, hold = max(cash, hold + p - fee), max(hold, cash - p)
    return cash

def max_profit_k(prices, k):
    """At most k transactions. buy[j]/sell[j] = best after j-th buy/sell."""
    buy = [-math.inf] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        for j in range(1, k + 1):
            buy[j] = max(buy[j], sell[j - 1] - p)
            sell[j] = max(sell[j], buy[j] + p)
    return sell[k]

P = [7, 1, 5, 3, 6, 4]
check("max_profit_one", max_profit_one(P) == 5)
check("max_profit_unlimited", max_profit_unlimited(P) == 7)
check("max_profit_cooldown", max_profit_cooldown([1, 2, 3, 0, 2]) == 3)
check("max_profit_fee", max_profit_fee([1, 3, 2, 8, 4, 9], 2) == 8)
check("max_profit_k", max_profit_k([3, 3, 5, 0, 0, 3, 1, 4], 2) == 6)

---
# 11. Backtracking

🧠 **Mental model:** DFS over a **decision tree**. At each node: *choose* an option, *explore* deeper, *un-choose* (undo). Prune branches that can't lead to a valid answer.

✏️ **Whiteboard sketch — subsets of [1,2,3]**
```
                      []
           /          |         \
         [1]         [2]        [3]
        /   \         |
     [1,2] [1,3]    [2,3]
       |
    [1,2,3]
Template:
def backtrack(path, choices):
    if is_solution(path): record(path[:])      # copy!
    for c in choices:
        if not valid(c): continue              # prune
        path.append(c)                         # choose
        backtrack(path, next_choices)          # explore
        path.pop()                             # un-choose
```

⏱️ Usually exponential: subsets O(n·2ⁿ), permutations O(n·n!). Say so — interviewers want to know you know.

🎯 Triggers: "all combinations / permutations / subsets", "generate all valid …", N-Queens, Sudoku, word search.

⚠️ **Gotchas:** appending `path` instead of `path[:]` (you'll get a list of empty lists); handling duplicates (sort first, skip `nums[i] == nums[i-1]` at the same depth).

In [ ]:
def subsets(nums):
    res, path = [], []
    def bt(start):
        res.append(path[:])
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return res

def subsets_with_dup(nums):
    nums = sorted(nums)
    res, path = [], []
    def bt(start):
        res.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:   # skip dup at same depth
                continue
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return res

def permutations(nums):
    res, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            res.append(path[:]); return
        for i, x in enumerate(nums):
            if used[i]: continue
            used[i] = True; path.append(x)
            bt()
            path.pop(); used[i] = False
    bt()
    return res

def combination_sum(cands, target):
    """Reuse allowed → recurse with i (not i+1). Sorted → break early (prune)."""
    cands = sorted(cands)
    res, path = [], []
    def bt(start, remain):
        if remain == 0:
            res.append(path[:]); return
        for i in range(start, len(cands)):
            if cands[i] > remain: break
            path.append(cands[i]); bt(i, remain - cands[i]); path.pop()
    bt(0, target)
    return res

def generate_parentheses(n):
    res = []
    def bt(s, opened, closed):
        if len(s) == 2 * n:
            res.append(s); return
        if opened < n: bt(s + "(", opened + 1, closed)
        if closed < opened: bt(s + ")", opened, closed + 1)
    bt("", 0, 0)
    return res

def n_queens(n):
    """Count solutions. Track used columns and both diagonals (r-c, r+c) in sets."""
    cols, d1, d2 = set(), set(), set()
    def bt(r):
        if r == n: return 1
        count = 0
        for c in range(n):
            if c in cols or r - c in d1 or r + c in d2:
                continue
            cols.add(c); d1.add(r - c); d2.add(r + c)
            count += bt(r + 1)
            cols.remove(c); d1.remove(r - c); d2.remove(r + c)
        return count
    return bt(0)

def word_search(board, word):
    R, C = len(board), len(board[0])
    def dfs(r, c, i):
        if i == len(word): return True
        if not (0 <= r < R and 0 <= c < C) or board[r][c] != word[i]: return False
        tmp, board[r][c] = board[r][c], "#"          # mark visited in place
        found = any(dfs(r + dr, c + dc, i + 1) for dr, dc in DIRS)
        board[r][c] = tmp                            # restore (un-choose)
        return found
    return any(dfs(r, c, 0) for r in range(R) for c in range(C))

check("subsets", len(subsets([1, 2, 3])) == 8 and [1, 3] in subsets([1, 2, 3]))
check("subsets_with_dup", len(subsets_with_dup([1, 2, 2])) == 6)
check("permutations", len(permutations([1, 2, 3])) == 6)
check("combination_sum", combination_sum([2, 3, 6, 7], 7) == [[2, 2, 3], [7]])
check("generate_parentheses", generate_parentheses(3) == ["((()))", "(()())", "(())()", "()(())", "()()()"])
check("n_queens", n_queens(4) == 2 and n_queens(8) == 92)
B = [list("ABCE"), list("SFCS"), list("ADEE")]
check("word_search", word_search(B, "ABCCED") and not word_search(B, "ABCB"))

---
# 12. Sorting, Selection & Intervals

🧠 **Mental model:**
| Algorithm | Time (avg / worst) | Space | Stable? | Say this |
|---|---|---|---|---|
| Merge sort | n log n / n log n | O(n) | ✅ | divide, sort halves, merge; great for linked lists & external sort |
| Quicksort | n log n / **n²** | O(log n) | ❌ | partition around pivot; random pivot avoids worst case |
| Heapsort | n log n / n log n | O(1) | ❌ | heapify then pop |
| Counting / radix | O(n + k) | O(k) | ✅ | small integer range |
| Timsort (Python) | n log n / n log n, **O(n) on nearly-sorted** | O(n) | ✅ | hybrid merge + insertion |
| Quickselect | **O(n)** / n² | O(1) | — | k-th element without full sort |

Lower bound for **comparison** sorting is Ω(n log n).

✏️ **Whiteboard sketch — intervals**
```
sort by start, then sweep:
[1,3]  ───
[2,6]   ──────        overlap if next.start <= cur.end → merge: end = max(ends)
[8,10]         ───
[15,18]                ────
→ [1,6] [8,10] [15,18]

"min meeting rooms" = max number of intervals overlapping at any time
 → sort by start, min-heap of end times (or sweep line of +1/-1 events)
```

In [ ]:
def merge_sort(a):
    if len(a) <= 1: return a
    mid = len(a) // 2
    left, right = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:            # <= keeps it stable
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    return out + left[i:] + right[j:]

def quicksort(a, lo=0, hi=None):
    """In-place, Lomuto partition, random pivot."""
    if hi is None: hi = len(a) - 1
    if lo >= hi: return a
    p = random.randint(lo, hi)
    a[p], a[hi] = a[hi], a[p]
    pivot, store = a[hi], lo
    for i in range(lo, hi):
        if a[i] < pivot:
            a[i], a[store] = a[store], a[i]
            store += 1
    a[store], a[hi] = a[hi], a[store]      # pivot into final place
    quicksort(a, lo, store - 1)
    quicksort(a, store + 1, hi)
    return a

def quickselect(a, k):
    """k-th smallest (0-indexed), average O(n). 3-way partition handles duplicates."""
    a = list(a)
    while True:
        pivot = random.choice(a)
        lows   = [x for x in a if x < pivot]
        pivots = [x for x in a if x == pivot]
        highs  = [x for x in a if x > pivot]
        if k < len(lows):
            a = lows
        elif k < len(lows) + len(pivots):
            return pivot
        else:
            k -= len(lows) + len(pivots)
            a = highs

def counting_sort(a, max_val):
    counts = [0] * (max_val + 1)
    for x in a: counts[x] += 1
    return [v for v, c in enumerate(counts) for _ in range(c)]

def sort_colors(nums):
    """Dutch national flag: 0s | 1s | unknown | 2s, one pass."""
    lo, i, hi = 0, 0, len(nums) - 1
    while i <= hi:
        if nums[i] == 0:
            nums[lo], nums[i] = nums[i], nums[lo]; lo += 1; i += 1
        elif nums[i] == 2:
            nums[hi], nums[i] = nums[i], nums[hi]; hi -= 1   # don't advance i
        else:
            i += 1
    return nums

data = [random.randint(-50, 50) for _ in range(200)]
check("merge_sort", merge_sort(data) == sorted(data))
check("quicksort", quicksort(data[:]) == sorted(data))
check("quickselect", quickselect(data, 17) == sorted(data)[17])
check("counting_sort", counting_sort([3, 1, 2, 1, 0], 3) == [0, 1, 1, 2, 3])
check("sort_colors", sort_colors([2, 0, 2, 1, 1, 0]) == [0, 0, 1, 1, 2, 2])

In [ ]:
def merge_intervals(intervals):
    out = []
    for s, e in sorted(intervals):
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out

def insert_interval(intervals, new):
    """Intervals already sorted & disjoint: left part, merged middle, right part."""
    s, e = new
    left  = [iv for iv in intervals if iv[1] < s]
    right = [iv for iv in intervals if iv[0] > e]
    mid = [iv for iv in intervals if not (iv[1] < s or iv[0] > e)]
    if mid:
        s, e = min(s, mid[0][0]), max(e, mid[-1][1])
    return left + [[s, e]] + right

def min_meeting_rooms(intervals):
    ends = []
    for s, e in sorted(intervals):
        if ends and ends[0] <= s:          # a room freed up
            heapq.heapreplace(ends, e)
        else:
            heapq.heappush(ends, e)
    return len(ends)

def erase_overlap_intervals(intervals):
    """Min removals so the rest don't overlap = n - max non-overlapping.
    Greedy: sort by END, keep anything that starts after the last kept end."""
    count, last_end = 0, -math.inf
    for s, e in sorted(intervals, key=lambda x: x[1]):
        if s >= last_end:
            count, last_end = count + 1, e
    return len(intervals) - count

check("merge_intervals", merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]])
check("insert_interval", insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]) == [[1, 2], [3, 10], [12, 16]])
check("min_meeting_rooms", min_meeting_rooms([[0, 30], [5, 10], [15, 20]]) == 2)
check("erase_overlap_intervals", erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1)

---
# 13. Bit Manipulation

🧠 **Mental model:** An integer is a row of switches. Tricks to know by heart:

| Expression | Meaning |
|---|---|
| `x & 1` | is x odd |
| `x & (x - 1)` | clear lowest set bit → `x & (x-1) == 0` ⇔ power of two (x > 0) |
| `x & -x` | isolate lowest set bit (Fenwick trees use this) |
| `a ^ a == 0`, `a ^ 0 == a` | XOR cancels pairs |
| `1 << k` | 2ᵏ / mask with bit k set |
| `(mask >> k) & 1` | is bit k set |
| `mask \| (1 << k)`, `mask & ~(1 << k)` | set / clear bit k |

⚠️ Python ints are **arbitrary precision** — no overflow, and negative numbers behave as infinite two's complement. To emulate 32-bit: `x & 0xFFFFFFFF`.

In [ ]:
def single_number(nums):
    """Every element appears twice except one → XOR everything."""
    r = 0
    for x in nums: r ^= x
    return r

def count_bits(x):
    """Brian Kernighan: one iteration per set bit."""
    c = 0
    while x:
        x &= x - 1; c += 1
    return c

def is_power_of_two(x):
    return x > 0 and x & (x - 1) == 0

def subsets_bitmask(nums):
    n = len(nums)
    return [[nums[i] for i in range(n) if mask >> i & 1] for mask in range(1 << n)]

def missing_number(nums):
    """0..n with one missing: XOR indices and values."""
    r = len(nums)
    for i, x in enumerate(nums): r ^= i ^ x
    return r

check("single_number", single_number([4, 1, 2, 1, 2]) == 4)
check("count_bits", count_bits(0b1011) == 3 == bin(0b1011).count("1"))
check("is_power_of_two", is_power_of_two(64) and not is_power_of_two(6))
check("subsets_bitmask", len(subsets_bitmask([1, 2, 3])) == 8)
check("missing_number", missing_number([3, 0, 1]) == 2)

---
# 14. Design Problems

These test whether you can **combine** structures to hit target complexities. Always start by stating the required ops and their target Big-O, then pick structures that satisfy each.

### LRU Cache — `get`/`put` in O(1)
- O(1) lookup → **hash map** key → node
- O(1) "move to most-recent" and "evict least-recent" → **doubly linked list**

```
 head ⇄ [k3] ⇄ [k1] ⇄ [k2] ⇄ tail        map: {k1: node, k2: node, k3: node}
 (most recent)          (least recent → evict tail.prev)
```
Sentinel head/tail nodes mean no `None` checks.

In [ ]:
class _Node:
    __slots__ = ("key", "val", "prev", "next")
    def __init__(self, key=0, val=0):
        self.key, self.val = key, val
        self.prev = self.next = None

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.map = {}
        self.head, self.tail = _Node(), _Node()
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, n):
        n.prev.next, n.next.prev = n.next, n.prev

    def _add_front(self, n):
        n.prev, n.next = self.head, self.head.next
        self.head.next.prev = n
        self.head.next = n

    def get(self, key):
        if key not in self.map: return -1
        n = self.map[key]
        self._remove(n); self._add_front(n)
        return n.val

    def put(self, key, val):
        if key in self.map:
            self._remove(self.map[key])
        n = _Node(key, val)
        self.map[key] = n
        self._add_front(n)
        if len(self.map) > self.cap:
            lru = self.tail.prev
            self._remove(lru)
            del self.map[lru.key]

class LRUCacheOD:
    """Same thing in 10 lines — mention it, but expect to be asked for the version above."""
    def __init__(self, capacity):
        self.cap, self.od = capacity, OrderedDict()
    def get(self, key):
        if key not in self.od: return -1
        self.od.move_to_end(key)
        return self.od[key]
    def put(self, key, val):
        self.od[key] = val
        self.od.move_to_end(key)
        if len(self.od) > self.cap:
            self.od.popitem(last=False)

for Cls in (LRUCache, LRUCacheOD):
    c = Cls(2)
    c.put(1, 1); c.put(2, 2)
    r1 = c.get(1)          # 1 → now most recent
    c.put(3, 3)            # evicts 2
    r2 = c.get(2)
    c.put(4, 4)            # evicts 1
    check(f"{Cls.__name__}", (r1, r2, c.get(1), c.get(3), c.get(4)) == (1, -1, -1, 3, 4))

In [ ]:
class MinStack:
    """push/pop/top/get_min all O(1): store (value, min_so_far) pairs."""
    def __init__(self): self.st = []
    def push(self, x): self.st.append((x, min(x, self.st[-1][1]) if self.st else x))
    def pop(self): self.st.pop()
    def top(self): return self.st[-1][0]
    def get_min(self): return self.st[-1][1]

class QueueWithTwoStacks:
    """Amortized O(1): pour `inbox` into `outbox` only when outbox is empty."""
    def __init__(self): self.inbox, self.outbox = [], []
    def push(self, x): self.inbox.append(x)
    def pop(self):
        if not self.outbox:
            while self.inbox: self.outbox.append(self.inbox.pop())
        return self.outbox.pop()

class SlidingWindowRateLimiter:
    """Allow at most `limit` requests per `window` seconds. Deque of timestamps."""
    def __init__(self, limit, window):
        self.limit, self.window, self.q = limit, window, deque()
    def allow(self, t):
        while self.q and self.q[0] <= t - self.window:
            self.q.popleft()
        if len(self.q) < self.limit:
            self.q.append(t); return True
        return False

class TimeMap:
    """Key-value store with timestamps; get(key, t) returns value at latest ts <= t.
    Timestamps arrive increasing → append + bisect."""
    def __init__(self): self.d = defaultdict(lambda: ([], []))
    def set(self, key, value, t):
        ts, vals = self.d[key]; ts.append(t); vals.append(value)
    def get(self, key, t):
        ts, vals = self.d[key]
        i = bisect.bisect_right(ts, t) - 1
        return vals[i] if i >= 0 else ""

ms = MinStack()
for x in [5, 3, 7, 2]: ms.push(x)
m1 = ms.get_min(); ms.pop()
check("MinStack", (m1, ms.get_min(), ms.top()) == (2, 3, 7))
q = QueueWithTwoStacks()
for x in [1, 2, 3]: q.push(x)
check("QueueWithTwoStacks", [q.pop(), q.pop()] == [1, 2])
rl = SlidingWindowRateLimiter(2, 10)
check("RateLimiter", [rl.allow(t) for t in [1, 2, 3, 11, 12, 13]] == [True, True, False, True, True, False])
tm = TimeMap(); tm.set("px", 100, 1); tm.set("px", 101, 5)
check("TimeMap", (tm.get("px", 0), tm.get("px", 3), tm.get("px", 9)) == ("", 100, 101))

---
# 15. 📈 Quant Corner — DSA in a trading context

Quant dev interviews love to dress classic structures up in market clothing. Recognize the costume:

| Market problem | Underlying DSA |
|---|---|
| Limit order book: best bid/ask, add/cancel/match | **heap (or sorted map) of price levels + dict of FIFO queues + dict id→order** |
| Rolling VWAP / moving average over time window | **deque + running sums** |
| Rolling max/min price over window | **monotonic deque** |
| Rolling median | **two heaps** (or sorted list + bisect) |
| Top-K symbols by traded volume | **hash map + heap** |
| "Price at time t" lookups | **sorted timestamps + bisect** |
| Best single buy/sell | **running min (Kadane-like)** |
| Currency arbitrage | **graph, negative cycle (Bellman-Ford on −log rates)** |

### Limit Order Book
```
       BIDS (buyers, max-heap)        ASKS (sellers, min-heap)
  price  queue (FIFO = time priority)   price  queue
  100.0  [o3:5, o7:2]  ← best bid       101.0  [o1:5]       ← best ask
   99.5  [o4:10]                        102.0  [o2:5, o9:1]
spread = best_ask - best_bid
Incoming BUY @102 qty 7 → crosses: fill o1 5 @101, then o2 2 @102 (price-time priority)
```
**Design decisions to say out loud:**
- Price levels: heap gives O(log n) best price; a cancelled/emptied level is removed **lazily** when it reaches the top.
- Within a level: `deque` for FIFO time priority.
- Cancel: O(1) by deleting from the id→order dict; dead ids are skipped lazily when they reach the front of the queue.
- In C++ you'd use `std::map` (red-black tree) per side; in Python, heaps + lazy deletion (or `sortedcontainers.SortedDict`).

In [ ]:
class OrderBook:
    def __init__(self):
        self.bids, self.asks = [], []                       # heaps: -price / price
        self.levels = {'B': {}, 'S': {}}                    # side -> price -> deque[order_id]
        self.orders = {}                                    # order_id -> [side, price, qty]

    def _heap(self, side):
        return self.bids if side == 'B' else self.asks

    def _best(self, side):
        heap, book = self._heap(side), self.levels[side]
        while heap:
            p = -heap[0] if side == 'B' else heap[0]
            dq = book.get(p)
            while dq and dq[0] not in self.orders:           # lazy-skip cancelled
                dq.popleft()
            if dq:
                return p
            heapq.heappop(heap)                              # level empty → drop it
            book.pop(p, None)
        return None

    def best_bid(self): return self._best('B')
    def best_ask(self): return self._best('S')

    def spread(self):
        b, a = self.best_bid(), self.best_ask()
        return None if b is None or a is None else a - b

    def add(self, oid, side, price, qty):
        """Limit order. Returns list of trades (taker_id, maker_id, price, qty)."""
        trades = []
        opp = 'S' if side == 'B' else 'B'
        crosses = (lambda p: p <= price) if side == 'B' else (lambda p: p >= price)
        while qty > 0:
            best = self._best(opp)
            if best is None or not crosses(best):
                break
            dq = self.levels[opp][best]
            maker_id = dq[0]
            maker = self.orders[maker_id]
            fill = min(qty, maker[2])
            trades.append((oid, maker_id, best, fill))      # trade at the RESTING price
            qty -= fill
            maker[2] -= fill
            if maker[2] == 0:
                dq.popleft()
                del self.orders[maker_id]
        if qty > 0:                                          # rest the remainder
            book = self.levels[side]
            if price not in book:
                book[price] = deque()
                heapq.heappush(self._heap(side), -price if side == 'B' else price)
            book[price].append(oid)
            self.orders[oid] = [side, price, qty]
        return trades

    def cancel(self, oid):
        return self.orders.pop(oid, None) is not None        # O(1); queue cleaned lazily

ob = OrderBook()
ob.add(1, 'S', 101, 5)
ob.add(2, 'S', 102, 5)
ob.add(3, 'B', 100, 3)
check("LOB best bid/ask", (ob.best_bid(), ob.best_ask(), ob.spread()) == (100, 101, 1))
trades = ob.add(4, 'B', 102, 7)
check("LOB matching price-time", trades == [(4, 1, 101, 5), (4, 2, 102, 2)])
check("LOB partial fill rests", ob.orders[2] == ['S', 102, 3] and ob.best_ask() == 102)
check("LOB cancel", ob.cancel(2) and ob.best_ask() is None)
ob.add(5, 'B', 100, 2)
check("LOB FIFO within level", ob.add(6, 'S', 99, 4) == [(6, 3, 100, 3), (6, 5, 100, 1)])

In [ ]:
class RollingVWAP:
    """VWAP over the last `window` seconds of trades. Each trade enters/leaves once → O(1) amortized."""
    def __init__(self, window):
        self.window, self.q = window, deque()
        self.pv = self.v = 0.0
    def on_trade(self, t, price, qty):
        self.q.append((t, price, qty))
        self.pv += price * qty; self.v += qty
        while self.q[0][0] <= t - self.window:
            _, p, q = self.q.popleft()
            self.pv -= p * q; self.v -= q
        return self.pv / self.v

def rolling_minmax(prices, k):
    """Rolling (min, max) over windows of size k with two monotonic deques. O(n)."""
    mn, mx, out = deque(), deque(), []
    for i, p in enumerate(prices):
        while mn and prices[mn[-1]] >= p: mn.pop()
        while mx and prices[mx[-1]] <= p: mx.pop()
        mn.append(i); mx.append(i)
        if mn[0] <= i - k: mn.popleft()
        if mx[0] <= i - k: mx.popleft()
        if i >= k - 1: out.append((prices[mn[0]], prices[mx[0]]))
    return out

def rolling_median(xs, k):
    """Sorted window + bisect: O(n·k) worst case due to list shifting, but simple and
    fast in practice. Mention the two-heap + lazy-deletion O(n log k) upgrade."""
    win, out = sorted(xs[:k]), []
    def med(): return win[k // 2] if k % 2 else (win[k // 2 - 1] + win[k // 2]) / 2
    out.append(med())
    for i in range(k, len(xs)):
        win.pop(bisect.bisect_left(win, xs[i - k]))
        bisect.insort(win, xs[i])
        out.append(med())
    return out

class TopKVolume:
    """Streaming volume per symbol; query top-k on demand. O(1) update, O(n log k) query."""
    def __init__(self): self.vol = Counter()
    def on_trade(self, sym, qty): self.vol[sym] += qty
    def top(self, k): return [s for s, _ in heapq.nlargest(k, self.vol.items(), key=lambda kv: kv[1])]

def has_arbitrage(rates):
    """rates[i][j] = units of j per unit of i. Arbitrage ⇔ product of a cycle > 1
    ⇔ sum of -log(rate) < 0 ⇔ negative cycle. Bellman-Ford from a virtual source."""
    n = len(rates)
    edges = [(i, j, -math.log(rates[i][j])) for i in range(n) for j in range(n) if i != j]
    dist = [0.0] * n                                   # virtual source to all
    for _ in range(n - 1):
        for u, v, w in edges:
            if dist[u] + w < dist[v] - 1e-12: dist[v] = dist[u] + w
    return any(dist[u] + w < dist[v] - 1e-12 for u, v, w in edges)

vw = RollingVWAP(window=10)
vw.on_trade(0, 100, 10); vw.on_trade(5, 102, 10)
check("RollingVWAP", vw.on_trade(12, 104, 20) == (102 * 10 + 104 * 20) / 30)
check("rolling_minmax", rolling_minmax([5, 3, 4, 8, 6, 1], 3) == [(3, 5), (3, 8), (4, 8), (1, 8)])
check("rolling_median", rolling_median([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1, -1, -1, 3, 5, 6])
tk = TopKVolume()
for s, q in [("AAPL", 100), ("MSFT", 300), ("AAPL", 250), ("TSLA", 50)]: tk.on_trade(s, q)
check("TopKVolume", tk.top(2) == ["AAPL", "MSFT"])
check("has_arbitrage", has_arbitrage([[1, 0.9, 0.8], [1.2, 1, 0.9], [1.3, 1.2, 1]]))
check("no arbitrage", not has_arbitrage([[1, 2, 4], [0.5, 1, 2], [0.25, 0.5, 1]]))

---
# 16. Whiteboard Playbook

### The 6-step script (say these out loud)
1. **Restate & clarify** — input size? sorted? duplicates? negatives? empty input? what to return if no answer?
2. **Examples** — work one normal example *and* one edge case by hand. Draw it.
3. **Brute force first** — state it and its complexity in one sentence. ("O(n²) by checking all pairs.")
4. **Optimize** — "the bottleneck is the inner search → a hash map makes that O(1)". Name the pattern.
5. **Code** — clean names, helper functions, leave room on the board (write small, start top-left).
6. **Test & analyze** — trace your example line by line, then edge cases; state final time & space.

### Edge-case checklist
`[]` · single element · all duplicates · negative numbers · zero · already sorted / reverse sorted · very large n (recursion depth!) · disconnected graph · cycle · k = 0 or k > n · integer division with negatives.

### When you're stuck
- Can I **sort** it first? (unlocks two pointers, binary search, greedy)
- Can a **hash map** remember something to avoid a re-scan?
- Is there a **monotonic** property? (binary search, monotonic stack)
- Can I think about the **last step** / **last element**? (DP)
- What if the input were tiny — can I **enumerate**? Then prune → backtracking.
- Is it secretly a **graph**? (states as nodes, moves as edges)

### 🔀 Pattern Router

| If the problem says… | Reach for… |
|---|---|
| sorted array, pair/triplet target | two pointers |
| contiguous subarray/substring, longest/shortest | sliding window |
| subarray sum = k, range sums | prefix sum + hash map |
| seen before / count / group | hash map / set / Counter |
| matching brackets, nested, undo | stack |
| next greater / smaller, span, histogram | monotonic stack |
| max/min in every window | monotonic deque |
| top-k, k-th largest, merge k sorted, median stream | heap(s) |
| sorted + find, or "minimum X such that feasible" | binary search (on answer) |
| shortest path, unweighted / grid | BFS |
| shortest path, weighted ≥ 0 | Dijkstra |
| dependencies / ordering / prerequisites | topological sort |
| connected components, dynamic connectivity | union-find (or DFS) |
| prefix / autocomplete / many words | trie |
| count ways / min cost / max value with overlapping subproblems | DP |
| all combinations / permutations / valid configurations | backtracking |
| overlapping intervals / scheduling | sort by start (merge) or by end (greedy) + heap |
| O(1) get/put with eviction | hash map + doubly linked list |
| "each element appears twice except…" | XOR |

### ⚠️ Python pitfalls that cost people offers
Run the cell below — each line demonstrates a classic trap.

In [ ]:
# 1. Aliased rows
bad = [[0] * 3] * 3; bad[0][0] = 1
check("[[0]*3]*3 aliases rows!", bad == [[1, 0, 0], [1, 0, 0], [1, 0, 0]])

# 2. Mutable default argument persists across calls
def append_bad(x, acc=[]):
    acc.append(x); return acc
append_bad(1)
check("mutable default arg leaks state", append_bad(2) == [1, 2])

# 3. Floor division vs truncation with negatives
check("-7 // 2 floors to -4", -7 // 2 == -4 and int(-7 / 2) == -3)
check("-7 % 3 is positive in Python", -7 % 3 == 2)

# 4. heapq with incomparable payloads → use a tiebreaker counter
h = []
counter = itertools.count()
heapq.heappush(h, (1, next(counter), {"order": "A"}))
heapq.heappush(h, (1, next(counter), {"order": "B"}))   # without counter: TypeError on dict < dict
check("heap tiebreaker", heapq.heappop(h)[2]["order"] == "A")

# 5. Shallow copy when recording backtracking paths
res, path = [], [1, 2]
res.append(path); path.append(3)
check("append(path) stores a reference, use path[:]", res[0] == [1, 2, 3])

# 6. Recursion limit (default ~1000) — know how to raise it or go iterative
import sys
check("recursion limit exists", sys.getrecursionlimit() >= 1000)

# 7. `is` vs `==`
check("use == for values", [1, 2] == [1, 2] and [1, 2] is not [1, 2])

# 8. String building in a loop: use list + join (strings are immutable)
check("join", "".join(["a", "b", "c"]) == "abc")

# 9. Sorting stability lets you sort by multiple keys in passes, or use tuple keys
people = [("bob", 25), ("amy", 30), ("cat", 25)]
check("tuple key sort", sorted(people, key=lambda p: (p[1], p[0])) == [("bob", 25), ("cat", 25), ("amy", 30)])

---
# 🎲 Self-Quiz

Run the cell below whenever you open this notebook. It picks 3 random prompts. **Answer on paper first** (pattern, complexity, key insight), then set `REVEAL = True` and re-run to check yourself.

In [ ]:
QUIZ = [
    ("Count subarrays summing to k (array has negatives).", "Prefix sum + hash map of prefix counts; seen[running-k]. O(n)/O(n). Sliding window fails with negatives."),
    ("Max of every window of size k.", "Monotonic decreasing deque of indices; pop front when out of window. O(n)."),
    ("Detect a cycle in a linked list and find its start.", "Floyd: slow/fast meet; reset one to head, step both by 1 → meet at start. O(n)/O(1)."),
    ("K-th largest element in a stream.", "Min-heap of size k; root is answer. O(log k) per add."),
    ("Course schedule: can you finish all courses?", "Topological sort (Kahn); cycle iff processed < n. O(V+E)."),
    ("Shortest path in a grid with walls.", "BFS from start, visited-on-enqueue, layer = distance. O(R·C)."),
    ("Minimum ship capacity to deliver within D days.", "Binary search on the answer in [max(w), sum(w)]; greedy feasibility check. O(n log sum)."),
    ("Longest substring without repeating characters.", "Sliding window + last-seen index map; jump left. O(n)."),
    ("Design LRU cache with O(1) get/put.", "Hash map key→node + doubly linked list with sentinels; move-to-front, evict tail.prev."),
    ("Number of connected components given edges.", "Union-find with path compression + union by size; components counter. ~O(E α(n))."),
    ("Validate a BST.", "DFS passing (lo, hi) bounds; or in-order must be strictly increasing. O(n)."),
    ("Running median of a stream.", "Two heaps: max-heap low half, min-heap high half; keep sizes balanced. O(log n) add, O(1) median."),
    ("Fewest coins to make an amount.", "Unbounded DP: dp[a] = 1 + min(dp[a-c]); dp[0]=0. O(amount·coins)."),
    ("Edit distance between two strings.", "2D DP over prefixes; insert ←, delete ↑, replace ↖. O(mn)."),
    ("Merge overlapping intervals.", "Sort by start; extend last if start <= last.end. O(n log n)."),
    ("Min meeting rooms.", "Sort by start; min-heap of end times; reuse room if heap[0] <= start. Answer = heap size."),
    ("Best time to buy/sell with cooldown.", "State machine DP: hold, sold, rest updated simultaneously each day. O(n)/O(1)."),
    ("Implement an order book with add/cancel/best bid.", "Per side: heap of prices + dict price→deque(order ids) + dict id→order; lazy deletion on cancel."),
    ("Detect currency arbitrage.", "Edge weights −log(rate); arbitrage ⇔ negative cycle; Bellman-Ford. O(V·E)."),
    ("Find the single number when all others appear twice.", "XOR all elements. O(n)/O(1)."),
    ("Largest rectangle in a histogram.", "Increasing monotonic stack; on pop, width = i - (new top) - 1. Sentinel 0 at end. O(n)."),
    ("All subsets of a set with duplicates.", "Sort; backtrack; skip nums[i]==nums[i-1] when i > start. O(n·2ⁿ)."),
    ("Search in a rotated sorted array.", "Binary search; one half is always sorted — test if target lies in it. O(log n)."),
    ("Autocomplete over a dictionary of words.", "Trie; walk prefix then DFS collecting words. O(L + output)."),
]

REVEAL = False
random.seed()                       # different picks every run
for i, (q, a) in enumerate(random.sample(QUIZ, 3), 1):
    print(f"Q{i}. {q}")
    print(f"    → {a}\n" if REVEAL else "    → (answer on paper, then set REVEAL = True)\n")

---
# 📅 Review Log

Copy this checklist into your notes and tick sections as you re-implement them **from memory**:

- [ ] Two pointers / sliding window / prefix sum — `min_window` from scratch
- [ ] Monotonic stack — `largest_rectangle` from scratch
- [ ] Linked list — reverse + Floyd cycle start, drawn on paper
- [ ] Trees — all three iterative traversals + `bst_delete`
- [ ] Heap — `MinHeap` sift up/down from scratch + `MedianFinder`
- [ ] Binary search — `first_true` template + one "search on answer" problem
- [ ] Graphs — BFS grid, Kahn's topo sort, Dijkstra, DSU
- [ ] Trie — insert/search/starts_with
- [ ] DP — coin change, LCS table drawn by hand, stock state machine
- [ ] Backtracking — subsets / permutations / N-Queens template
- [ ] Sorting — merge sort, quicksort partition, quickselect, intervals
- [ ] Design — LRU with doubly linked list
- [ ] Quant — order book walkthrough explained out loud in < 5 min

**Spacing suggestion:** day 1 → day 3 → day 7 → the night before each interview: only the Pattern Router + Self-Quiz.

Good luck — you've got this. 🚀